<a href="https://colab.research.google.com/github/UmymaM/ml-dl-cv-fundamentals/blob/main/emotion-detection/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import os, cv2, torch, torchvision
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import datasets,transforms
import torch.nn as nn
import zipfile as zf

In [31]:
!kaggle datasets download -d ananthu017/emotion-detection-fer

Dataset URL: https://www.kaggle.com/datasets/ananthu017/emotion-detection-fer
License(s): CC0-1.0
emotion-detection-fer.zip: Skipping, found more recently modified local copy (use --force to force download)


In [32]:
zipfile=zf.ZipFile('emotion-detection-fer.zip') #unzipping the file
zipfile.extractall()
zipfile.close()

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [34]:
processor=AutoImageProcessor.from_pretrained("google/efficientnet-b0")
def transform_fn(image):
    return processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)

EDA

In [35]:
path="/content/train"
for file in os.listdir(path):
    folder_path=os.path.join(path,file)
    print(f"{file} :{len(os.listdir(folder_path))}")
    images=os.listdir(folder_path)
    image_path=os.path.join(folder_path,images[0])
    img=Image.open(image_path)
    img.show()
    print(img.size)

happy :7215
(48, 48)
surprised :3171
(48, 48)
sad :4830
(48, 48)
neutral :4965
(48, 48)
angry :3995
(48, 48)
disgusted :436
(48, 48)
fearful :4097
(48, 48)


In [36]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, TensorDataset

In [37]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [38]:
# dataset=datasets.ImageFolder(root=path, transform=train_transform)
dataset=datasets.ImageFolder(root=path, transform=transform_fn)

In [39]:
dataset.classes

['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']

In [40]:
labels=dataset.targets
len(labels)

28709

In [41]:
from collections import Counter
class_counts=Counter(labels)
class_counts

Counter({0: 3995, 1: 436, 2: 4097, 3: 7215, 4: 4965, 5: 4830, 6: 3171})

In [42]:
#inversing frequency as weights
# class_weights=1.0/class_counts
class_weights={cls: 1/count for cls,count in class_counts.items()}
class_weights

{0: 0.00025031289111389235,
 1: 0.0022935779816513763,
 2: 0.000244081034903588,
 3: 0.0001386001386001386,
 4: 0.0002014098690835851,
 5: 0.00020703933747412008,
 6: 0.000315357931251971}

In [43]:
sample_weights=[class_weights[label]for label in labels]
len(sample_weights)

28709

In [44]:
sampler=WeightedRandomSampler(weights=sample_weights,num_samples=len(sample_weights),replacement=True)

In [45]:
train_loader=DataLoader(dataset=dataset,batch_size=32,sampler=sampler)

In [46]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

processor = AutoImageProcessor.from_pretrained("google/efficientnet-b0")
model = AutoModelForImageClassification.from_pretrained("google/efficientnet-b0")

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

In [47]:
model.classifier.in_features

1280

In [48]:
# model.classifier=nn.Sequential(
#     nn.Dropout(p=0.3),
#     nn.Linear(1280,7)
# )

In [49]:
model.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 7)
)
model = model.to(device)

In [50]:
# model.classifier

In [51]:
for param in model.parameters():
    param.requires_grad=False #freezing backbone

In [52]:
# model.classifier = nn.Sequential(
#     nn.Dropout(p=0.3),
#     nn.Linear(1280,7)
# )

In [53]:
for param in model.classifier.parameters():
    param.requires_grad=True #unfreezing the classifier

In [54]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=model.classifier.parameters(),lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [55]:
for epoch in range(15):  #phase 1 of training(frozen backbone+training classifier)
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=images)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    print(f"Epoch {epoch+1:2d}/15 | "f"Loss: {total_loss/len(train_loader):.4f} | "f"Acc: {100*correct/total:.1f}%")

Epoch  1/15 | Loss: 1.6999 | Acc: 33.8%
Epoch  2/15 | Loss: 1.5788 | Acc: 39.4%
Epoch  3/15 | Loss: 1.5317 | Acc: 40.9%
Epoch  4/15 | Loss: 1.4912 | Acc: 42.8%
Epoch  5/15 | Loss: 1.4559 | Acc: 43.7%
Epoch  6/15 | Loss: 1.4287 | Acc: 44.9%
Epoch  7/15 | Loss: 1.3958 | Acc: 46.0%
Epoch  8/15 | Loss: 1.3815 | Acc: 47.0%
Epoch  9/15 | Loss: 1.3703 | Acc: 46.9%
Epoch 10/15 | Loss: 1.3739 | Acc: 46.8%
Epoch 11/15 | Loss: 1.3448 | Acc: 47.9%
Epoch 12/15 | Loss: 1.3467 | Acc: 47.9%
Epoch 13/15 | Loss: 1.3270 | Acc: 48.6%
Epoch 14/15 | Loss: 1.3277 | Acc: 48.7%
Epoch 15/15 | Loss: 1.3143 | Acc: 49.3%


In [56]:
for param in model.parameters():
  param.requires_grad=True  #unfreezing the backbone

In [57]:
optimizer=torch.optim.Adam(params=model.parameters(),lr=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [62]:
# phase 2 of training
for epoch in range(10):
  model.train()
  total_loss, correct, total = 0,0,0
  for images,labels in train_loader:
    images,labels=images.to(device),labels.to(device)
    optimizer.zero_grad()
    output=model(pixel_values=images)
    loss=criterion(output.logits,labels)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
    correct += (output.logits.argmax(1) == labels).sum().item()
    total += labels.size(0)
  scheduler.step()
  print(f"Epoch {epoch+1:2d}/15 | "f"Loss: {total_loss/len(train_loader):.4f} | "f"Acc: {100*correct/total:.1f}%")


Epoch  1/15 | Loss: 1.2630 | Acc: 51.4%
Epoch  2/15 | Loss: 1.1916 | Acc: 54.5%
Epoch  3/15 | Loss: 1.1555 | Acc: 56.0%
Epoch  4/15 | Loss: 1.1203 | Acc: 57.2%
Epoch  5/15 | Loss: 1.0749 | Acc: 59.0%
Epoch  6/15 | Loss: 1.0515 | Acc: 59.7%
Epoch  7/15 | Loss: 1.0365 | Acc: 60.2%
Epoch  8/15 | Loss: 1.0241 | Acc: 60.9%
Epoch  9/15 | Loss: 1.0122 | Acc: 61.4%
Epoch 10/15 | Loss: 0.9941 | Acc: 62.2%


In [ ]:
def evalaute(model,loader):
  model.eval()
  total_loss,correct,total=0,0,0
  with torch.no_grad():
    for images,labels in loader:
      images, labels=images.to(device),labels.to(device)
      output=model(pixel_values=images)
      predictions=outputs.logit.argmax(1)
      correct+=(predictions==labels).sum().item()
      total+=labels.size(0)
  return 100*correct/total